In [20]:
!pip install catboost

     |████████████████████████████████| 76.2 MB 268 kB/s             


In [62]:
import pandas as pd
data = pd.read_csv('/opt/neurothink/neuro-a78463d5b0564a0aa3e8b7f1ec721e8e/atlas-higgs-challenge-2014-v2.csv')


In [63]:
train_data = data.loc[data['KaggleSet'] == 't']
public_test_data = data.loc[data['KaggleSet'] == 'b']
private_test_data = data.loc[data['KaggleSet'] == 'v']

features = [f for f in train_data if f not in ['EventId', 'Label', 'Weight', 'KaggleWeight', 'KaggleSet']]
target = 'Label'

In [64]:
train_data[features].head()

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,197.760,...,-0.277,258.733,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497
1,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,125.157,...,-1.916,164.546,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226
2,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,197.814,...,-2.186,260.414,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251
3,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,75.968,...,0.060,86.062,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000
4,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,57.983,...,-0.871,53.131,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000


In [88]:
import catboost
from catboost import CatBoostClassifier
import numpy as np
import time

start = time.time()

"""
cb = CatBoostClassifier(iterations=1000,
                        learning_rate=0.1,
                        depth=5, task_type='CPU',
                        random_state=1,
                        verbose=False)
"""

cb = CatBoostClassifier(iterations=10000,
                        depth=5, task_type='CPU',
                        random_state=1,
                        verbose=False)

cb.fit(train_data[features], train_data[target])

end = time.time()

In [89]:
y_pred = cb.predict(public_test_data[features])
from sklearn.metrics import accuracy_score

cb_acc = accuracy_score(public_test_data[target], y_pred)

print("Acuracy score for CatBoost VALIDATION: ", np.round(cb_acc, 2))
print('CPU training time for CatBoost (in Seconds):', (end - start))

Acuracy score for CatBoost VALIDATION:  0.84
CPU training time for CatBoost (in Seconds): 213.30077147483826


In [74]:
y_pred = cb.predict(private_test_data[features])
cb_acc = accuracy_score(private_test_data[target], y_pred)

print("Acuracy score for CatBoost TEST: ", np.round(cb_acc, 2))

Acuracy score for CatBoost TEST:  0.84


In [55]:
y_pred

array(['b', 's', 's', ..., 'b', 's', 'b'], dtype=object)

In [68]:
import math

def AMS(s, b):
    """ Approximate Median Significance defined as:
        AMS = sqrt(
                2 { (s + b + b_r) log[1 + (s/(b+b_r))] - s}
              )        
    where b_r = 10, b = background, s = signal, log is natural logarithm """
    
    br = 10.0
    radicand = 2 *( (s+b+br) * math.log (1.0 + s/(b+br)) -s)
    if radicand < 0:
        print('radicand is negative. Exiting')
        exit()
    else:
        return math.sqrt(radicand)
    
def compute_performance(logfile, setname, y_pred, y_gold, y_weight):
    s = 0
    b = 0
    tp = 0
    fp = 0
    for i,y in enumerate(y_pred):
        if y == 's':
            if y_gold.iloc[i] == 's':
                s += y_weight.iloc[i]
                tp += 1
            else:
                b += y_weight.iloc[i]
                fp += 1

    out1 = f"{setname}: (S,B) = ({s:.3f},{b:.3f}), (TP,FP) = {tp}, {fp}"
    print(out1)
    logfile.write(out1 + '\n')

    outstring = f"{setname}: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}"
    print(outstring)
    logfile.write(outstring + '\n')

In [42]:
from datetime import datetime
now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')

In [90]:
y_pred = cb.predict(train_data[features])
y_gold = train_data[target]
y_weight = train_data['KaggleWeight']
compute_performance(logfile, "TRAINING", y_pred, y_gold, y_weight)
y_pred = cb.predict(public_test_data[features])
y_gold = public_test_data[target]
y_weight = public_test_data['KaggleWeight']
compute_performance(logfile, "VALIDATION", y_pred, y_gold, y_weight)
y_pred = cb.predict(private_test_data[features])
y_gold = private_test_data[target]
y_weight = private_test_data['KaggleWeight']
compute_performance(logfile, "TEST", y_pred, y_gold, y_weight)

TRAINING: (S,B) = (492.540,24296.385), (TP,FP) = 64388, 14359
TRAINING: (S,B) = (492.540,24296.385), AMS=3.149, Unweighted Precision = 0.818
VALIDATION: (S,B) = (480.649,27429.600), (TP,FP) = 24905, 6551
VALIDATION: (S,B) = (480.649,27429.600), AMS=2.893, Unweighted Precision = 0.792
TEST: (S,B) = (481.096,27062.236), (TP,FP) = 112945, 29142
TEST: (S,B) = (481.096,27062.236), AMS=2.915, Unweighted Precision = 0.795
